# Synthetic PDU telemetry generator

Writes **second-level** PDU telemetry to `<catalog>.<schema>.<bronze_table>` so the prototype is demoable without a live feed.

* Layout: `num_racks` racks x `shelves` shelves x `positions` positions (default 4 x 5 x 5 = 100 PDUs). `unit_id` = `Rxx-Sy-Pz`.
* Normal behaviour: ~240 V with slow per-rack drift, 30-40 C inlet temperature rising with shelf height, 11-16 A per unit.
* Excursions (deterministic per unit and time window, so reruns are reproducible):
  * per-unit voltage sags/spikes of 9-26 V lasting 20-120 s (mix of WARN-band and BREACH-band),
  * rack-wide 30 s feeder dips of -14 V,
  * thermal ramps of +8 to +28 C over 20 minutes,
  * one chronically hot unit (`R03-S5-P4`, steady WARN) and one flaky unit (`R02-S3-P2`, drops off for 15 min in some hours).
* Data-quality noise: ~0.03% null voltage readings and ~0.02% `-40 C` sensor faults (dropped by pipeline expectations).
* Incremental: each run appends from the last `event_ts` up to now (capped at `backfill_hours`), then deletes rows older than `retention_days`.

The first run creates the schema (if missing) and the bronze table, clustered by `(unit_id, event_ts)` for fast single-unit troubleshooting queries.

In [0]:
dbutils.widgets.text("catalog", "mdk_demo_catalog")
dbutils.widgets.text("schema", "pdu_monitoring")
dbutils.widgets.text("bronze_table", "bronze_pdu_telemetry")
dbutils.widgets.text("backfill_hours", "6")
dbutils.widgets.text("retention_days", "7")
dbutils.widgets.text("num_racks", "4")
dbutils.widgets.text("shelves", "5")
dbutils.widgets.text("positions", "5")

CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
BRONZE = dbutils.widgets.get("bronze_table")
BACKFILL_HOURS = int(dbutils.widgets.get("backfill_hours"))
RETENTION_DAYS = int(dbutils.widgets.get("retention_days"))
NUM_RACKS = int(dbutils.widgets.get("num_racks"))
SHELVES = int(dbutils.widgets.get("shelves"))
POSITIONS = int(dbutils.widgets.get("positions"))

import re
for name, val in {"catalog": CATALOG, "schema": SCHEMA, "bronze_table": BRONZE}.items():
    if not re.fullmatch(r"[A-Za-z0-9_]+", val):
        raise ValueError(f"Invalid identifier for {name}: {val!r}")

FQN = f"`{CATALOG}`.`{SCHEMA}`.`{BRONZE}`"
print(FQN)

In [0]:
import math
from pyspark.sql import functions as F

UNITS_PER_RACK = SHELVES * POSITIONS
HOT_UNIT = "R03-S5-P4"    # runs ~12 C hotter than its neighbours (steady WARN)
FLAKY_UNIT = "R02-S3-P2"  # occasionally stops reporting (stale data)
TWO_PI = 2 * math.pi


def u(*parts):
    """Deterministic pseudo-random number in [0, 1) derived from the given columns."""
    return F.pmod(F.xxhash64(*parts), F.lit(1_000_000)) / F.lit(1_000_000.0)


def units_df():
    return (
        spark.range(NUM_RACKS * UNITS_PER_RACK)
        .withColumn("rack_no", (F.col("id") / UNITS_PER_RACK).cast("int") + 1)
        .withColumn("shelf", ((F.col("id") % UNITS_PER_RACK) / POSITIONS).cast("int") + 1)
        .withColumn("position", (F.col("id") % POSITIONS).cast("int") + 1)
        .withColumn("rack_id", F.format_string("R%02d", "rack_no"))
        .withColumn("unit_id", F.format_string("R%02d-S%d-P%d", "rack_no", "shelf", "position"))
        # Last rack holds higher-density PDUs (used for per-model threshold overrides).
        .withColumn("unit_model", F.when(F.col("rack_no") == NUM_RACKS, "PDU-HD-32A").otherwise("PDU-STD-24A"))
        .drop("id", "rack_no")
    )


def telemetry(start_s: int, end_s: int):
    """One row per unit per second in [start_s, end_s) (epoch seconds)."""
    unit, rack, ts = F.col("unit_id"), F.col("rack_id"), F.col("ts")
    df = units_df().crossJoin(spark.range(start_s, end_s).withColumnRenamed("id", "ts"))

    w10, t10 = (ts / 600).cast("long"), ts % 600     # 10-minute voltage-event windows
    w20, t20 = (ts / 1200).cast("long"), ts % 1200   # 20-minute thermal-event windows

    # Voltage: nominal 240 V + rack offset/drift + noise + per-unit sags/spikes + rack feeder dips
    rack_offset = (u(rack, F.lit("roff")) - 0.5) * 3
    rack_wave = 1.5 * F.sin(TWO_PI * ts / 1200 + u(rack, F.lit("rph")) * TWO_PI)
    has_v = u(unit, w10, F.lit("v")) < 0.035
    v_start = 60 + F.floor(u(unit, w10, F.lit("vs")) * 400)
    v_dur = 20 + F.floor(u(unit, w10, F.lit("vd")) * 100)
    v_mag = 9 + 17 * u(unit, w10, F.lit("vm"))
    v_sign = F.when(u(unit, w10, F.lit("vsg")) < 0.6, -1.0).otherwise(1.0)
    in_v = has_v & (t10 >= v_start) & (t10 < v_start + v_dur)
    rack_sag = (u(rack, w10, F.lit("rsag")) < 0.012) & (t10 >= 300) & (t10 < 330)
    voltage = (240 + rack_offset + rack_wave + F.randn() * 0.6
               + F.when(in_v, v_sign * v_mag).otherwise(0.0)
               + F.when(rack_sag, F.lit(-14.0)).otherwise(0.0))

    # Temperature: base per unit, warmer higher in the rack, daily cycle, noise, thermal ramps
    base_t = (30 + 8 * u(unit, F.lit("bt")) + 0.6 * (F.col("shelf") - 1)
              + F.when(unit == HOT_UNIT, 12.0).otherwise(0.0))
    has_t = u(unit, w20, F.lit("t")) < 0.02
    t_amp = 8 + 20 * u(unit, w20, F.lit("ta"))
    temperature = (base_t + 1.5 * F.sin(TWO_PI * ts / 86400) + F.randn() * 0.15
                   + F.when(has_t, t_amp * F.sin(math.pi * t20 / 1200)).otherwise(0.0))

    current = 11 + 5 * u(unit, F.lit("cur")) + 0.8 * F.sin(TWO_PI * ts / 900) + F.randn() * 0.1
    humidity = 42 + 6 * F.sin(TWO_PI * ts / 86400 + u(rack, F.lit("hph")) * TWO_PI) + F.randn() * 0.3

    offline = ((unit == FLAKY_UNIT)
               & (u(unit, (ts / 3600).cast("long"), F.lit("off")) < 0.25)
               & ((ts % 3600) < 900))

    df = (df.where(~offline)
            .withColumn("voltage_v", voltage)
            .withColumn("temperature_c", temperature)
            .withColumn("current_a", current)
            .withColumn("humidity_pct", humidity)
            .withColumn("power_kw", F.col("voltage_v") * F.col("current_a") * 0.97 / 1000)
            .withColumn("glitch", F.rand()))
    # Data-quality noise for pipeline expectations
    df = (df.withColumn("voltage_v", F.when(F.col("glitch") < 0.0003, F.lit(None).cast("double")).otherwise(F.col("voltage_v")))
            .withColumn("temperature_c", F.when((F.col("glitch") >= 0.0003) & (F.col("glitch") < 0.0005), F.lit(-40.0)).otherwise(F.col("temperature_c"))))

    return df.select(
        "unit_id", "rack_id", "unit_model", "shelf", "position",
        F.timestamp_seconds("ts").alias("event_ts"),
        F.round("voltage_v", 2).alias("voltage_v"),
        F.round("current_a", 2).alias("current_a"),
        F.round("power_kw", 3).alias("power_kw"),
        F.round("temperature_c", 2).alias("temperature_c"),
        F.round("humidity_pct", 1).alias("humidity_pct"),
        F.current_timestamp().alias("ingest_ts"),
    )

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG}`.`{SCHEMA}`")
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {FQN} (
  unit_id       STRING    NOT NULL COMMENT 'PDU identifier, Rxx-Sy-Pz',
  rack_id       STRING    NOT NULL COMMENT 'Rack identifier, Rxx',
  unit_model    STRING    COMMENT 'PDU model (drives per-model threshold overrides)',
  shelf         INT       COMMENT 'Shelf (row) in the rack, 1 = bottom',
  position      INT       COMMENT 'Position (column) on the shelf, 1 = left',
  event_ts      TIMESTAMP NOT NULL COMMENT 'Reading timestamp (UTC, 1 s resolution)',
  voltage_v     DOUBLE    COMMENT 'Input voltage (V)',
  current_a     DOUBLE    COMMENT 'Load current (A)',
  power_kw      DOUBLE    COMMENT 'Active power (kW)',
  temperature_c DOUBLE    COMMENT 'PDU inlet temperature (C)',
  humidity_pct  DOUBLE    COMMENT 'Relative humidity (%)',
  ingest_ts     TIMESTAMP COMMENT 'Time the row was written'
)
CLUSTER BY (unit_id, event_ts)
COMMENT 'Second-level PDU telemetry (bronze). Synthetic for the prototype.'
""")

In [0]:
import time

now_s = int(time.time())
last_ts = spark.sql(f"SELECT unix_timestamp(max(event_ts)) AS t FROM {FQN}").first()["t"]
earliest_allowed = now_s - BACKFILL_HOURS * 3600
start_s = earliest_allowed if last_ts is None else max(int(last_ts) + 1, earliest_allowed)

if start_s >= now_s:
    rows_written = 0
    print("Bronze table is already up to date.")
else:
    telemetry(start_s, now_s).write.mode("append").saveAsTable(f"{CATALOG}.{SCHEMA}.{BRONZE}")
    rows_written = (now_s - start_s) * NUM_RACKS * UNITS_PER_RACK  # upper bound (flaky unit drops some)
    print(f"Appended ~{rows_written:,} rows covering {(now_s - start_s) / 60:.1f} minutes.")

In [0]:
# Only issue a DELETE when something is actually past retention, to keep the Delta log small.
min_ts = spark.sql(f"SELECT unix_timestamp(min(event_ts)) AS t FROM {FQN}").first()["t"]
cutoff_s = now_s - RETENTION_DAYS * 86400
if min_ts is not None and min_ts < cutoff_s:
    spark.sql(f"DELETE FROM {FQN} WHERE event_ts < timestamp_seconds({cutoff_s})")
    print(f"Deleted rows older than {RETENTION_DAYS} days.")
else:
    print("Nothing past retention.")

In [0]:
display(spark.sql(f"""
  SELECT count(*) AS rows, count(DISTINCT unit_id) AS units, count(DISTINCT rack_id) AS racks,
         min(event_ts) AS first_ts, max(event_ts) AS last_ts
  FROM {FQN}
"""))